# M2: conditional low-rank Gaussian copula

M2 tests whether features already available at the forecast origin predict residual spatial dependence. For each entity it maps the frozen feature vector $v_{k,\tau}^{(i)}$ to factor loadings and a uniqueness scale:

$$ (\lambda_{k,\tau}^{(i)},b_{k,\tau}^{(i)})=f_\theta(v_{k,\tau}^{(i)}),\qquad \sigma_{k,\tau}^{(i)}=\operatorname{softplus}(b_{k,\tau}^{(i)})+\sigma_{\min}. $$

Writing the complete loading matrix as $\Lambda$, M2 uses $\Sigma=\Lambda\Lambda^\mathsf T+\operatorname{diag}(\sigma^2)$, normalized to $R$. The same shared network is applied to each entity; it does not contextualize entity features before producing its parameters.

In [ ]:
from pathlib import Path
import sys, numpy as np
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
from simcast.config import SimcastConfig, deep_merge, load_config
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

CONFIG_FILE = 'configs/powertech2027/transformer.yaml'
OVERRIDES = ()
TRAIN_IF_MISSING = False
RUN_EVALUATION = False
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm2_conditional_low_rank'
base = load_config(PROJECT_ROOT / CONFIG_FILE, overrides=OVERRIDES)
config = SimcastConfig.model_validate(deep_merge(base.model_dump(mode='python'), {'dependence': {'method': 'conditional_low_rank'}}))
CACHE_DIR = PROJECT_ROOT / config.output.cache_dir / (config.output.cache_name or f'liander2024_{config.data.entity_type}')
library = load_pit_library(CACHE_DIR, access='training'); ds = library.dataset
entity_ids = [str(x) for x in ds.entity_id.values]; K_g = len(entity_ids)
assert entity_ids == config.protocol.ordered_entity_ids and K_g == config.protocol.entity_count
print(f'group={config.data.entity_type}, K_g={K_g}, rank={config.dependence.conditional_low_rank.latent_rank}')

## The scientific covariates

The cache provides a Chronos output-patch representation, the finite quantile shape, median, log spread, within-patch lead position, and optional coordinates. Scalar normalization is fitted on the training partition only. Thus $v_{k,\tau}^{(i)}$ is measurable with respect to $\mathcal I^{(i)}$; neither the future observation nor a test-derived normalization may enter it.

Although M2 is equivariant to reordering, the analysis always supplies exactly the declared ordered group $\mathcal E_g$. The loading axes themselves are rotation non-identifiable, so interpretation belongs to $R$, not to an individual coordinate of $\lambda$.

In [ ]:
embeddings = np.asarray(ds['forecast_embedding'].values)
quantiles = np.asarray(ds['quantile_prediction'].values)
scores = np.asarray(ds['pit_z'].values)
valid = np.isfinite(scores).all(axis=1)
print('embeddings [origin, entity, patch, feature]:', embeddings.shape)
print('quantiles  [origin, entity, lead, level]:', quantiles.shape)
print('complete training cases:', int((valid & (np.asarray(ds.split.values)[:, None] == 'train')).sum()))
assert embeddings.shape[1] == quantiles.shape[1] == K_g

## Fitting and evaluation

M2 minimizes the Gaussian-copula pseudo-negative log likelihood on complete training vectors and selects a checkpoint by validation pseudo-NLL. The optional cell below calls the same public functions as the CLI. It is deliberately disabled so that reading this notebook never creates a run or touches an active experiment.

In [ ]:
run_dir = OUTPUT_DIR / 'conditional_low_rank'
if TRAIN_IF_MISSING and not run_dir.exists():
    run_dir = train_from_config(config, cache_dir=CACHE_DIR, output_dir=run_dir)
if RUN_EVALUATION:
    if not run_dir.exists(): raise FileNotFoundError('Set TRAIN_IF_MISSING=True or choose an existing M2 run.')
    evaluate_from_config(config, methods=('conditional_low_rank',), method_runs={'conditional_low_rank': run_dir}, cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR / 'evaluation')
else:
    print('Read-only mode: no M2 training or evaluation artifact is written.')

In [ ]:
import pandas as pd
from IPython.display import display
metrics_file = OUTPUT_DIR / 'evaluation' / 'metrics_by_lead.csv'
if metrics_file.is_file():
    metrics = pd.read_csv(metrics_file)
    display(metrics.groupby('method', as_index=False).mean(numeric_only=True))
    metrics.pivot(index='lead', columns='method', values='mean_pinball').plot(title='M2 aggregate pinball loss by lead')
else:
    print('No notebook evaluation table yet. Read-only inspection does not create one.')